In [1]:
import sys
import site
site.addsitedir(site.getusersitepackages())
import scipy
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import sktime
import sklearn
from scipy.io import loadmat

In [2]:
## Directory of all .MAT files
matdata_dir = "C:/Users/kafumbej/Desktop/MAP2026/matfiles/matfiles"

## Disposition
disposition = pd.read_csv("C:/Users/kafumbej/Desktop/MAP2026/disposition.csv")

## All analyses should only use files marked "Reduced"
reduced_disp = disposition[disposition['Reduced'] == "X"]

## Add .mat to DAQs so they can be read from the mat data directory, reset index to facilitate looping
reduced_disp = reduced_disp.assign(MatName = reduced_disp['DaqName'].str[:-4] + '.mat').reset_index(0, drop = True)

In [ ]:
## Empty final df
final_df = pd.DataFrame(columns=['Subject', 'Drive', 'LPupil_Diameter', 'RPupil_Diameter', 'Lateral_Deviation', 'Sample', 'Blink_Duration',
                                 'Left_Gaze_Dir_X', 'Left_Gaze_Dir_Y', 'Left_Gaze_Dir_Z', 'Right_Gaze_Dir_X', 'Right_Gaze_Dir_Y',
                                 'Right_Gaze_Dir_Z', 'Gaze_Pitch', 'Gaze_Yaw', 'Left_Eyelid_Closed', 'Right_Eyelid_Closed',
                                 'Vehicle_Speed', 'Brake_Pedal_Force', 'Steering_Wheel_Angle', 'Steering_Wheel_Angle_Rate', 'Accelerator_Pedal_Position',
                                 'Head_Pos_X', 'Head_Pos_Y', 'Head_Pos_Z', 'Blink_Counter', 'Blink_Frequency'])
 
for i in range(0, reduced_disp.shape[0] - 1):
   
    ## Read current mat file
    fname = matdata_dir + "/" + reduced_disp['MatName'][i]
    try:
        data = loadmat(fname)
        print(f"Successfully read {fname}")
    except FileNotFoundError:
        print(f"Skipping {fname}: File not found.")
   
    ## Keep only "rural straight" scenario
    logstream = data['elemDataI']['SCC_LogStreams'][0][0][:,1]
    ruralstraight_idx = logstream == 311
   
    ## Vars to keep (just lane dev and pupil diam for now)
   
    ## Full lat pos series
    lat_pos_series = data['elemDataI']['SCC_Lane_Deviation'][0][0][:,1]
    lat_pos_series = lat_pos_series[ruralstraight_idx][::10].flatten()
    nstep = lat_pos_series.shape[0]
   
    ## Full left pupil series
    try:
        left_pupil_series = data['elemDataI']['Output_FovioDMEResults_dme_core_pupil_left_pupil_diameter_mm'][0][0]
        left_pupil_series = left_pupil_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        left_pupil_series = np.repeat(99, nstep)
 
    ## Full right pupil series
    try:
        right_pupil_series = data['elemDataI']['Output_FovioDMEResults_dme_core_pupil_right_pupil_diameter_mm'][0][0]
        right_pupil_series = right_pupil_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        right_pupil_series = np.repeat(99, nstep)
 
    ## Blink counter series
    try:
        blink_counter_series = data['elemDataI']['Output_FovioDMEResults_dme_core_eyelid_blink_counter'][0][0]
        blink_counter_series = blink_counter_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        blink_counter_series = np.repeat(99, nstep)
 
    ## Blink frequency series
    try:
        blink_frequency_series = data['elemDataI']['Output_FovioDMEResults_dme_core_eyelid_blink_frequency'][0][0]
        blink_frequency_series = blink_frequency_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        blink_counter_series = np.repeat(99, nstep)
   
    ## Head pos x series
    try:
        head_pos_x_series = data['elemDataI']['Output_FovioDMEResults_dme_core_head_head_pose_translation_x'][0][0]
        head_pos_x_series = head_pos_x_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        head_pos_x_series = np.repeat(99, nstep)
   
    ## Head pos y series
    try:
        head_pos_y_series = data['elemDataI']['Output_FovioDMEResults_dme_core_head_head_pose_translation_y'][0][0]
        head_pos_y_series = head_pos_y_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        head_pos_y_series = np.repeat(99, nstep)
 
    ## Head pos z series
    try:
        head_pos_z_series = data['elemDataI']['Output_FovioDMEResults_dme_core_head_head_pose_translation_z'][0][0]
        head_pos_z_series = head_pos_z_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        head_pos_z_series = np.repeat(99, nstep)
 
     ## Blink duration series
    try:
        blink_duration_series = data['elemDataI']['Output_FovioDMEResults_dme_core_eyelid_average_blink_duration_u'][0][0]
        blink_duration_series = blink_duration_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        blink_duration_series = np.repeat(99, nstep)
 
    ## Left Gaze direction x series
    try:
        lgaze_dir_x_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_left_eye_gaze_direction_x'][0][0]
        lgaze_dir_x_series = lgaze_dir_x_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        lgaze_dir_x_series = np.repeat(99, nstep)
   
    ## Left Gaze direction y series
    try:
        lgaze_dir_y_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_left_eye_gaze_direction_y'][0][0]
        lgaze_dir_y_series = lgaze_dir_y_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        lgaze_dir_y_series = np.repeat(99, nstep)
   
     ## Left Gaze direction z series
    try:
        lgaze_dir_z_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_left_eye_gaze_direction_z'][0][0]
        lgaze_dir_z_series = lgaze_dir_z_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        lgaze_dir_z_series = np.repeat(99, nstep)
 
 
    ## Right Gaze direction x series
    try:
        rgaze_dir_x_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_right_eye_gaze_direction_x'][0][0]
        rgaze_dir_x_series = rgaze_dir_x_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        rgaze_dir_x_series = np.repeat(99, nstep)
   
    ## Right Gaze direction y series
    try:
        rgaze_dir_y_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_right_eye_gaze_direction_y'][0][0]
        rgaze_dir_y_series = rgaze_dir_y_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        rgaze_dir_y_series = np.repeat(99, nstep)
   
     ## Right Gaze direction z series
    try:
        rgaze_dir_z_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_right_eye_gaze_direction_z'][0][0]
        rgaze_dir_z_series = rgaze_dir_z_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        rgaze_dir_z_series = np.repeat(99, nstep)
 
      ## Gaze direction pitch
    try:
        gaze_pitch_series = data['elemDataI']['Output_FovioDMEResults_dgaze_unified_gaze_direction_deg_pitch_d'][0][0]
        gaze_pitch_series = gaze_pitch_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        gaze_pitch_series = np.repeat(99, nstep)
 
      ## Gaze direction yaw
    try:
        gaze_yaw_series = data['elemDataI']['Output_FovioDMEResults_dgaze_unified_gaze_direction_deg_yaw_deg'][0][0]
        gaze_yaw_series = gaze_yaw_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        gaze_yaw_series = np.repeat(99, nstep)
 
      ## Left eyelid closed boolean
    try:
        right_closed_series = data['elemDataI']['Output_FovioDMEResults_dme_core_eyelid_right_eyelid_closed'][0][0]
        right_closed_series = right_closed_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        right_closed_series = np.repeat(99, nstep)
 
      ## right eyelid closed boolean
    try:
        left_closed_series = data['elemDataI']['Output_FovioDMEResults_dme_core_eyelid_left_eyelid_closed'][0][0]
        left_closed_series = left_closed_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        left_closed_series = np.repeat(99, nstep)
 
    ## Vehicle Speed
    try:
        veh_speed_series = data['elemDataI']['VDS_Veh_Speed'][0][0]
        veh_speed_series = veh_speed_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        veh_speed_series = np.repeat(99, nstep)
 
 
    ## Brake Pedal Force
    try:
        brake_force_series = data['elemDataI']['CFS_Brake_Pedal_Force'][0][0]
        brake_force_series = brake_force_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        brake_force_series = np.repeat(99, nstep)
 
    ## Steering wheel angle
    try:
        steer_angle = data['elemDataI']['CFS_Steering_Wheel_Angle'][0][0]
        steer_angle = steer_angle[ruralstraight_idx][::10].flatten()
    except ValueError:
        steer_angle = np.repeat(99, nstep)
 
   
    ## Steering wheel angle rate
    try:
        steer_angle_rate = data['elemDataI']['CFS_Steering_Wheel_Angle_Rate'][0][0]
        steer_angle_rate = steer_angle_rate[ruralstraight_idx][::10].flatten()
    except ValueError:
        steer_angle_rate = np.repeat(99, nstep)
 
    ## Accelerator Pedal Position
    try:
        acc_pedal_series = data['elemDataI']['CFS_Accelerator_Pedal_Position'][0][0]
        acc_pedal_series = acc_pedal_series[ruralstraight_idx][::10].flatten()
    except ValueError:
        acc_pedal_series = np.repeat(99, nstep)
 
    ## Assemble DF w/ Subject and Drive meta data
    temp_df = pd.DataFrame({'Subject': np.repeat(reduced_disp['Subject'][i],  nstep),
                 'Drive': np.repeat(reduced_disp['DriveN'][i],  nstep),
                 'RPupil_Diameter': right_pupil_series,
                 'LPupil_Diameter': left_pupil_series,
                 'Lateral_Deviation': lat_pos_series,
                 'Blink_Duration': blink_duration_series,
                 'Left_Gaze_Dir_X': lgaze_dir_x_series,
                 'Left_Gaze_Dir_Y': lgaze_dir_y_series,
                 'Left_Gaze_Dir_Z': lgaze_dir_z_series,
                 'Right_Gaze_Dir_X': rgaze_dir_x_series,
                 'Right_Gaze_Dir_Y': rgaze_dir_y_series,
                 'Right_Gaze_Dir_Z': rgaze_dir_z_series,
                 'Gaze_Pitch': gaze_pitch_series,
                 'Gaze_Yaw': gaze_yaw_series,
                 'Left_Eyelid_Closed': left_closed_series,
                 'Right_Eyelid_Closed': right_closed_series,
                 'Vehicle_Speed': veh_speed_series,
                 'Brake_Pedal_Force': brake_force_series,
                 'Steering_Wheel_Angle': steer_angle,
                 'Steering_Wheel_Angle_Rate': steer_angle_rate,
                 'Accelerator_Pedal_Position': acc_pedal_series,
                'Head_Pos_X': head_pos_x_series,
                'Head_Pos_Y': head_pos_y_series,
                'Head_Pos_Z': head_pos_z_series,
                'Blink_Counter': blink_counter_series,
                'Blink_Frequency': blink_frequency_series
 
    })
   
    ## Attach "Sample" for each 60 sec sample. Note - dividing by 360 bc 10 frames per sec
    temp_df['Sample'] = 1 + temp_df.index // 360
   
    ## Discard any samples that aren't length 360
    temp_df = temp_df[temp_df.groupby('Sample')['Sample'].transform('size') == 360]
   
    ## Append the temp df to the current final df
    final_df = pd.concat([final_df, temp_df], ignore_index=True)

Successfully read C:\Users\kafumbej\Desktop\MAP2026\matfiles\matfiles/NADS_IMPAIR_2A_20230821082155.mat
Skipping C:\Users\kafumbej\Desktop\MAP2026\matfiles\matfiles/NADS_IMPAIR_1A_20230821103336.mat: File not found.
Successfully read C:\Users\kafumbej\Desktop\MAP2026\matfiles\matfiles/NADS_IMPAIR_3A_20230821111658.mat
Successfully read C:\Users\kafumbej\Desktop\MAP2026\matfiles\matfiles/NADS_IMPAIR_1B_20230821122105.mat
Successfully read C:\Users\kafumbej\Desktop\MAP2026\matfiles\matfiles/NADS_IMPAIR_2B_20230821133736.mat
Skipping C:\Users\kafumbej\Desktop\MAP2026\matfiles\matfiles/NADS_IMPAIR_1A_20230817082432.mat: File not found.
Successfully read C:\Users\kafumbej\Desktop\MAP2026\matfiles\matfiles/NADS_IMPAIR_3A_20230817111633.mat
Successfully read C:\Users\kafumbej\Desktop\MAP2026\matfiles\matfiles/NADS_IMPAIR_3A_RS1_20230817112239.mat
Successfully read C:\Users\kafumbej\Desktop\MAP2026\matfiles\matfiles/NADS_IMPAIR_2A_20230817120321.mat
Successfully read C:\Users\kafumbej\Desktop\

In [61]:
# Sanity check values for each column
for col_name, vals in final_df.items():
    print(f"{col_name}: {(vals != 99).sum()}")

Subject: 649440
Drive: 649440
LPupil_Diameter: 632160
RPupil_Diameter: 632160
Lateral_Deviation: 649440
Sample: 649440
Blink_Duration: 632160
Left_Gaze_Dir_X: 632160
Left_Gaze_Dir_Y: 632160
Left_Gaze_Dir_Z: 632160
Right_Gaze_Dir_X: 632160
Right_Gaze_Dir_Y: 632160
Right_Gaze_Dir_Z: 632160
Gaze_Pitch: 632160
Gaze_Yaw: 632160


c:\ProgramData\anaconda3\Lib\site-packages\pandas\core\ops\array_ops.py:75: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)


Left_Eyelid_Closed: 632160
Right_Eyelid_Closed: 632160
Vehicle_Speed: 649440
Brake_Pedal_Force: 649440
Steering_Wheel_Angle: 649440
Steering_Wheel_Angle_Rate: 649440
Accelerator_Pedal_Position: 649440
Head_Pos_X: 632160
Head_Pos_Y: 632160
Head_Pos_Z: 632160
Blink_Counter: 632154
Blink_Frequency: 649440
